In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage
import os

In [ ]:
load_dotenv(override=True)

In [ ]:
# Note: Ensure 'CV Pr Mohamed YOUSSFI V9.pdf' exists in the directory
pdf_path = "CV Pr Mohamed YOUSSFI V9.pdf"
if not os.path.exists(pdf_path):
    print(f"Warning: {pdf_path} not found. Please provide the PDF file.")
else:
    loader = PyPDFLoader(pdf_path)
    tokennizer = tiktoken.encoding_for_model("gpt-4o-mini")

    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name=tokennizer.name, chunk_size=300, chunk_overlap=20
    )

    chunks = loader.load_and_split(splitter)
    embedding_model = OpenAIEmbeddings()

    vector_store = Chroma.from_documents(
        documents=chunks, embedding=embedding_model, collection_name="cv_data_collection"
    )

    retriever = vector_store.as_retriever(kwargs={"k": 10})

In [ ]:
@tool
def retriever_tool(query: str) -> str:
    """
    Permet de chercher des informations sur des candidats :
    - Nom, Prénom, Diplômes
    - Expériences
    - Compétences
    """
    relevant_chunks = retriever.invoke(query)
    context_list = [d.page_content for d in relevant_chunks]
    context = ". ".join(context_list)
    return context

@tool
def get_company_infos(company_name: str):
    """
    Consulter des informations sur l'entreprise donnée
    """
    return {"company_name": company_name, "domain": "IT", "turnover": 120_870_000}

In [ ]:
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = "Répond à la question de l'utilisateur en utilisant les tools fournis"

agent_executor = create_react_agent(
    model=llm,
    tools=[retriever_tool, get_company_infos],
    state_modifier=system_prompt,
)

In [ ]:
response = agent_executor.invoke({"messages": [HumanMessage(content="Qui est Mohamed YOUSSFI ?")]})
print(response["messages"][-1].content)